# Encoder-only models: BERT and variants

An encoder-only LM reads both left and right context, then predicts or classifies from contextual states.

Encoder-only models sit on Transformer attention and change the mask and objective. Bidirectional context makes masked-token prediction and classification natural, but it does not create a left-to-right generator. Save a copy to Drive to edit.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(91)


def softmax(logits):
    values = np.asarray(logits, dtype=float)
    shifted = values - values.max(axis=-1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=-1, keepdims=True)


def cosine_scores(query, candidates):
    query = np.asarray(query, dtype=float)
    candidates = np.asarray(candidates, dtype=float)
    query_norm = np.linalg.norm(query) + 1e-12
    candidate_norms = np.linalg.norm(candidates, axis=1) + 1e-12
    return candidates @ query / (candidate_norms * query_norm)


def encoder_ladder():
    return [
        {
            "name": "D1 masked four-token sentence",
            "tokens": ["ads", "[MASK]", "drive", "growth"],
            "mask_positions": [1],
            "labels": ["clicks"],
            "segment_ids": [0, 0, 0, 0],
            "target": 0,
        },
        {
            "name": "D2 clean MLM set",
            "tokens": ["fresh", "creative", "[MASK]", "strong", "clicks"],
            "mask_positions": [2],
            "labels": ["wins"],
            "segment_ids": [0, 0, 0, 0, 0],
            "target": 0,
        },
        {
            "name": "D3 repeated tokens and segment flips",
            "tokens": ["brand", "launch", "[MASK]", "brand", "offer", "[MASK]"],
            "mask_positions": [2, 5],
            "labels": ["video", "segment"],
            "segment_ids": [0, 0, 0, 1, 1, 1],
            "target": 0,
        },
        {
            "name": "D4 tiny intent corpus",
            "tokens": ["[CLS]", "pause", "low", "quality", "campaign", "[SEP]"],
            "mask_positions": [3],
            "labels": ["quality"],
            "segment_ids": [0, 0, 0, 0, 0, 0],
            "target": 1,
        },
        {
            "name": "D5 long pair with heavy masking",
            "tokens": ["[CLS]", "account", "asks", "why", "[MASK]", "dropped", "[SEP]", "creative", "has", "low", "[MASK]", "and", "weak", "[MASK]"],
            "mask_positions": [4, 10, 13],
            "labels": ["ctr", "quality", "hook"],
            "segment_ids": [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
            "target": 1,
        },
    ]


VOCAB = ["clicks", "ctr", "quality", "hook", "wins", "segment", "video", "budget"]
TOKEN_VECTORS = {token: RNG.normal(size=6) for token in VOCAB + ["ads", "drive", "growth", "fresh", "creative", "strong", "brand", "launch", "offer", "pause", "low", "campaign", "account", "asks", "why", "dropped", "has", "and", "weak", "[CLS]", "[SEP]"]}
SEGMENT_VECTORS = {0: np.zeros(6), 1: np.array([0.15, -0.1, 0.05, 0.0, 0.1, -0.05])}

## The concept, built once: bidirectional masked prediction

Encoder-only models use full self-attention and predict masked slots:
$$H=\mathrm{Encoder}(X+P+S),\; p(x_m\mid x_{\setminus m})=\mathrm{softmax}(h_m W_o).$$
For the lesson's toy case, full attention over $T=4$ tokens has $4\times4=16$ visible pairs.

In [ ]:
def masked_encoder_predict(tokens, mask_positions, segment_ids, candidate_words):
    token_matrix = []
    for index, token in enumerate(tokens):
        base = TOKEN_VECTORS.get(token, np.zeros(6))
        position = np.full(6, index / max(1, len(tokens)))
        segment = SEGMENT_VECTORS[int(segment_ids[index])]
        token_matrix.append(base + 0.05 * position + segment)
    x = np.vstack(token_matrix)
    scores = x @ x.T / math.sqrt(x.shape[1])
    attention = softmax(scores)
    states = attention @ x
    candidate_matrix = np.vstack([TOKEN_VECTORS[word] for word in candidate_words])
    predictions = []
    probabilities = []
    for pos in mask_positions:
        logits = states[pos] @ candidate_matrix.T
        probs = softmax(logits)
        predictions.append(candidate_words[int(np.argmax(probs))])
        probabilities.append(probs)
    return {
        "attention": attention,
        "states": states,
        "predictions": predictions,
        "probabilities": np.vstack(probabilities),
        "visible_pairs": len(tokens) * len(tokens),
    }

Now plug in the exact lesson numbers. The softmax of logits $[2,1,0]$ should be approximately $[0.665,0.245,0.090]$, 15% masking of 20 tokens should create 3 supervised positions, and a 3-label head on $h_{CLS}\in\mathbb{R}^{768}$ should use 2307 parameters.

In [ ]:
d1 = encoder_ladder()[0]
built = masked_encoder_predict(d1["tokens"], d1["mask_positions"], d1["segment_ids"], VOCAB[:3])
lesson_probs = softmax(np.array([2.0, 1.0, 0.0]))
supervised_positions = int(0.15 * 20)
classifier_parameters = 768 * 3 + 3
segment_shape_before = (4, 768)
segment_shape_after = (4, 768)

assert built["visible_pairs"] == 16
assert np.allclose(np.round(lesson_probs, 3), np.array([0.665, 0.245, 0.090]))
assert supervised_positions == 3
assert classifier_parameters == 2307
assert segment_shape_before == segment_shape_after

print("visible pairs", built["visible_pairs"])
print("lesson softmax", np.round(lesson_probs, 3))
print("masked labels from 20 tokens", supervised_positions)
print("3-label CLS head parameters", classifier_parameters)
print("segment-added shape", segment_shape_after)

## The dataset ladder

D1-D5 increases ambiguity, task realism, and masking pressure while keeping the same encoder-style method.

In [ ]:
ladder = encoder_ladder()
for rung in ladder:
    shape = (len(rung["tokens"]), 6)
    print(rung["name"], "shape", shape, "masks", len(rung["mask_positions"]), "sample", rung["tokens"][:6])

In [ ]:
rows = []
for rung in ladder:
    result = masked_encoder_predict(rung["tokens"], rung["mask_positions"], rung["segment_ids"], VOCAB)
    correct = 0
    for predicted, label in zip(result["predictions"], rung["labels"]):
        correct += int(predicted == label)
    accuracy = correct / max(1, len(rung["labels"]))
    rows.append({
        "name": rung["name"],
        "context": len(rung["tokens"]),
        "accuracy": accuracy,
        "attention": result["attention"],
        "predictions": result["predictions"],
    })
for row in rows:
    print(f"{row['name']}: context={row['context']} accuracy={row['accuracy']:.3f} predictions={row['predictions']}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for idx, row in enumerate(rows):
    ax = axes[0, idx]
    image = ax.imshow(row["attention"], vmin=0.0, vmax=row["attention"].max(), cmap="Blues")
    ax.set_title(f"D{idx + 1}")
    ax.set_xlabel("key")
    ax.set_ylabel("query")
    axes[1, idx].bar(["accuracy"], [row["accuracy"]], color="steelblue")
    axes[1, idx].set_ylim(0, 1)
    axes[1, idx].set_title(f"len={row['context']}")
fig.colorbar(image, ax=axes[0, :].ravel().tolist(), shrink=0.7)
fig.suptitle("Encoder-only outputs per rung and accuracy vs context")
plt.figure(figsize=(6, 3))
plt.plot([row["context"] for row in rows], [row["accuracy"] for row in rows], marker="o")
plt.xlabel("context length")
plt.ylabel("masked-token / label accuracy")
plt.ylim(-0.05, 1.05)
plt.grid(True)

## Pitfall on D5: encoder-only states are not a free-form generator

The lesson warns that $p(x_m\mid x_{\setminus m})$ is a masked-position objective, not a left-to-right product. On D5, pretending it can continue text gives an invalid repeated fill. The fix is to keep the task as masked prediction or classification.

In [ ]:
hard = ladder[-1]
bad_continuation = []
for step in range(4):
    result = masked_encoder_predict(hard["tokens"], hard["mask_positions"], hard["segment_ids"], VOCAB)
    bad_continuation.append(result["predictions"][0])
fixed = masked_encoder_predict(hard["tokens"], hard["mask_positions"], hard["segment_ids"], VOCAB)
fixed_accuracy = sum(int(a == b) for a, b in zip(fixed["predictions"], hard["labels"])) / len(hard["labels"])
print("wrong free-form continuation", bad_continuation)
print("fixed masked-task predictions", fixed["predictions"])
print("fixed masked-task accuracy", round(fixed_accuracy, 3))

## Evaluate it + practice

            - Metric: masked-token or label accuracy; compare to the no-skill baseline, majority label or most common mask fill.
            - Cheap sanity check: attention rows sum to 1 and masks see both sides.
            - Ablation: remove segment or position vectors and watch ambiguity increase.
            - Failure signals: generation-like loops repeat fills or ignore positions.
            - Practice: Add one repeated token to D3 and inspect attention.
- Practice: Change the mask rate on D5 and recompute accuracy.
- Practice: Swap segment IDs and explain which predictions change.